In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from tqdm import tqdm
from IPython.display import Video, display

%matplotlib inline

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('running on', device)

SAVE_FIGURES = False  # set to True to write the figures used on the slides to ./figures
def savefig(name):
    if SAVE_FIGURES:
        os.makedirs('figures', exist_ok=True)
        plt.savefig(f'figures/{name}.png', dpi=150, bbox_inches='tight')

def save_video(frames, name, fps=4, width=700):
    """write a list of RGB uint8 frames to name.mp4 (in figures/ if SAVE_FIGURES) and show it"""
    folder = 'figures' if SAVE_FIGURES else '.'
    os.makedirs(folder, exist_ok=True)
    filename, tmp = f'{folder}/{name}.mp4', f'{folder}/{name}_mp4v.mp4'
    h, w = frames[0].shape[:2]
    writer = cv2.VideoWriter(tmp, cv2.VideoWriter_fourcc(*'mp4v'), fps, (w, h))
    for frame in frames:
        writer.write(cv2.cvtColor(frame, cv2.COLOR_RGB2BGR))
    writer.release()
    # OpenCV writes mp4v; re-encode to H.264, which browsers and PowerPoint play
    os.system(f'ffmpeg -loglevel error -y -i {tmp} -vf "pad=ceil(iw/2)*2:ceil(ih/2)*2:color=white" '
              f'-vcodec libx264 -pix_fmt yuv420p -crf 18 {filename}')
    os.remove(tmp)
    display(Video(filename, embed=True, width=width))

def grab(fig):
    """the current content of a matplotlib figure as an RGB array (a video frame)"""
    fig.canvas.draw()
    return np.asarray(fig.canvas.buffer_rgba())[..., :3].copy()

In [ ]:
%%sh
# Download the data - you need to do this only once
wget --no-verbose --output-document=image_cc_3.jpg https://github.com/chrirupp/cv_course/raw/main/data/image_cc_3.jpg
wget --no-verbose --output-document=image.jpg https://github.com/chrirupp/cv_course/raw/main/data/image.jpg

# Lecture 18 - 3D and World Models

**3D from a single image.** A feed-forward network predicts metric depth (and the focal length) for every pixel; with the pinhole model (Lecture 14) we lift the image to a 3D point cloud and look at it from new viewpoints. The holes that appear are exactly what a generative (world) model has to fill in.

**A toy world model.** A world model predicts what we will observe next, given what we do. We train one on a tiny simulated room and use it for the three ideas of the lecture: the future is uncertain (regression predicts the blurred mean, a generative model samples), errors compound when the model is rolled out on its own predictions, and a world model can be used to *plan*.

Run this notebook on Google Colab (see the [readme](https://github.com/chrirupp/cv_course)). Cells that load large pretrained models or train are faster with a GPU runtime (*Edit > Notebook settings > T4 GPU*).

In [ ]:
def load_rgb(filename, width=768):
    image = cv2.cvtColor(cv2.imread(filename), cv2.COLOR_BGR2RGB)
    return cv2.resize(image, (width, int(round(image.shape[0] * width / image.shape[1]))), interpolation=cv2.INTER_AREA)

images = {'quad': load_rgb('image_cc_3.jpg'), 'pots': load_rgb('image.jpg')}

## Monocular metric depth

Depth Pro (Apple, 2024) is a ViT-based network trained on a large mix of real and synthetic data. From one image it predicts *metric* depth (in metres) and also the camera's focal length, so we do not need to know which lens took the photo. A single image has no absolute scale (Lecture 14): the metres come from what the network has learned about the world, such as how big doors, people and plant pots usually are.

In [ ]:
from transformers import DepthProImageProcessor, DepthProForDepthEstimation
dtype = torch.float16 if device == 'cuda' else torch.float32
depth_processor = DepthProImageProcessor.from_pretrained('apple/DepthPro-hf')
depth_model = DepthProForDepthEstimation.from_pretrained('apple/DepthPro-hf', torch_dtype=dtype).to(device).eval()

def predict_depth(image):
    """metric depth (H, W) in metres and the focal length in pixels"""
    inputs = depth_processor(images=image, return_tensors='pt')
    with torch.no_grad():
        out = depth_model(pixel_values=inputs['pixel_values'].to(device, dtype))
    post = depth_processor.post_process_depth_estimation(out, target_sizes=[image.shape[:2]])[0]
    return post['predicted_depth'].float().cpu().numpy(), float(post['focal_length'])

depths, focal = {}, {}
for name, image in images.items():
    depths[name], focal[name] = predict_depth(image)

fig, axs = plt.subplots(2, 2, figsize=(16, 11))
for row, name in zip(axs, images):
    row[0].imshow(images[name])
    row[0].set_title('input image', fontsize=15)
    im = row[1].imshow(depths[name], cmap='turbo_r', vmin=0, vmax=np.percentile(depths[name], 98))
    plt.colorbar(im, ax=row[1], label='depth (m)', fraction=0.03)
    fov = 2 * np.degrees(np.arctan(images[name].shape[1] / 2 / focal[name]))
    row[1].set_title(f'predicted depth; predicted field of view {fov:.0f}°', fontsize=15)
    for ax in row:
        ax.axis('off')
plt.tight_layout()
savefig('3d_depth')
plt.show()

## Lifting the image to 3D

Pixel $(u, v)$ with depth $z$ is the 3D point $\mathbf{X} = z\, K^{-1} (u, v, 1)^\top$ in the camera's coordinates: the inverse of the projection from Lecture 14, with the focal length from Depth Pro in $K$. The sky has no depth, so we remove it first with a semantic segmentation network (SegFormer, Lecture 11).

To look at the points from a new camera, we project them again and keep the nearest point per pixel (a z-buffer). Each point covers the area its pixel covered in the photo: points that come closer to the new camera get bigger. Pixels that no point reaches are left empty.

In [ ]:
from transformers import AutoImageProcessor, AutoModelForSemanticSegmentation
seg_name = 'nvidia/segformer-b0-finetuned-ade-512-512'
seg_processor = AutoImageProcessor.from_pretrained(seg_name)
seg_model = AutoModelForSemanticSegmentation.from_pretrained(seg_name).to(device).eval()

def sky_mask(image):
    with torch.no_grad():
        logits = seg_model(**seg_processor(images=image, return_tensors='pt').to(device)).logits
    labels = F.interpolate(logits, size=image.shape[:2], mode='bilinear', align_corners=False).argmax(1)[0]
    return (labels == seg_model.config.label2id['sky']).cpu().numpy()

def intrinsics(image, f):
    h, w = image.shape[:2]
    return np.array([[f, 0, w / 2], [0, f, h / 2], [0, 0, 1]])

def unproject(depth, K):
    h, w = depth.shape
    u, v = np.meshgrid(np.arange(w) + 0.5, np.arange(h) + 0.5)
    rays = np.stack([u, v, np.ones_like(u)], -1).reshape(-1, 3) @ np.linalg.inv(K).T
    return rays * depth.reshape(-1, 1)

def render(points, colors, K, R, t, shape, background=(255, 255, 255)):
    """z-buffer rendering of a coloured point cloud given in the original camera's frame (world = original camera).
    A point covers about as many pixels as its pixel did in the photo: (original depth / new depth) pixels wide."""
    h, w = shape
    P = torch.as_tensor(points, dtype=torch.float32, device=device)
    col = torch.as_tensor(colors, device=device)
    Xc = P @ torch.as_tensor(R.T, dtype=torch.float32, device=device) + torch.as_tensor(t, dtype=torch.float32, device=device)
    z = Xc[:, 2]
    front = z > 0.05
    P, col, Xc, z = P[front], col[front], Xc[front], z[front]
    uv = Xc @ torch.as_tensor(K.T, dtype=torch.float32, device=device)
    u, v = uv[:, 0] / z, uv[:, 1] / z
    size = (2.0 * P[:, 2] / z).ceil().clamp(1, 6).long()
    x0, y0 = u.floor().long() - size // 2, v.floor().long() - size // 2
    zbuf = torch.full((h * w,), float('inf'), device=device)
    pix, dep, src = [], [], []
    for dy in range(6):
        for dx in range(6):
            m = (size > dx) & (size > dy)
            x, y = x0[m] + dx, y0[m] + dy
            ok = (x >= 0) & (x < w) & (y >= 0) & (y < h)
            pix.append((y * w + x)[ok]); dep.append(z[m][ok]); src.append(torch.nonzero(m)[:, 0][ok])
    pix, dep, src = torch.cat(pix), torch.cat(dep), torch.cat(src)
    zbuf.scatter_reduce_(0, pix, dep, 'amin')
    win = dep <= zbuf[pix]                                  # the nearest point wins the pixel
    out = torch.tensor(background, dtype=torch.uint8, device=device).repeat(h * w, 1)
    out[pix[win]] = col[src[win]]
    img, hit = out.view(h, w, 3).cpu().numpy(), torch.isfinite(zbuf).view(h, w).cpu().numpy()
    # close the pixel-sized gaps between neighbouring samples (not the holes behind objects): fill an empty pixel with
    # the mean of its covered neighbours if most of them are covered
    for _ in range(2):
        n = cv2.boxFilter(hit.astype(np.float32), -1, (3, 3), normalize=False)
        gap = ~hit & (n >= 5)
        s = cv2.boxFilter(img.astype(np.float32) * hit[..., None], -1, (3, 3), normalize=False)
        img[gap] = (s[gap] / n[gap, None]).astype(np.uint8)
        hit = hit | gap
    return img, hit

def look_at(C, target):
    """rotation and translation of a camera at C looking at target, image 'up' = world -y (as in the photo)"""
    z = (target - C) / np.linalg.norm(target - C)
    x = np.cross([0, 1, 0], z)
    x = x / np.linalg.norm(x)
    R = np.stack([x, np.cross(z, x), z])
    return R, -R @ C

def draw_camera(img, K, R, t, K0, shape0, d, color=(0, 0, 0)):
    """draw the original camera (at the origin, looking along +z) as a frustum of depth d into a rendered view"""
    h0, w0 = shape0
    corners = np.array([[0, 0, 1], [w0, 0, 1], [w0, h0, 1], [0, h0, 1]]) @ np.linalg.inv(K0).T * d
    pts = np.vstack([np.zeros(3), corners]) @ R.T + t
    uv = (pts @ K.T)[:, :2] / (pts @ K.T)[:, 2:]
    p = [tuple(int(round(c)) for c in q) for q in uv]
    for a, b in [(0, 1), (0, 2), (0, 3), (0, 4), (1, 2), (2, 3), (3, 4), (4, 1)]:
        cv2.line(img, p[a], p[b], color, 3, cv2.LINE_AA)
    return img

clouds = {}
for name, image in images.items():
    K = intrinsics(image, focal[name])
    keep = ~sky_mask(image).ravel()
    clouds[name] = (unproject(depths[name], K)[keep], image.reshape(-1, 3)[keep], K)
    print(f'{name}: {keep.sum()} points, {100 * (1 - keep.mean()):.0f}% of the pixels are sky')

### Flying up from the photo

The camera starts where the photo was taken (the first frame is the photo itself) and rises to look down on the scene; the red frustum is the original camera. We only get the surface the camera saw (2.5D): walls have no back and there is nothing behind the people or the pots.

In [ ]:
def lift_video(name, end, frames=40, hold=14):
    points, colors, K = clouds[name]
    image = images[name]
    target = np.array([0, 0, np.median(points[:, 2])])           # on the optical axis: the first frame is the photo
    video = []
    for k in range(frames):
        s = 0.5 - 0.5 * np.cos(np.pi * k / (frames - 1))          # smooth start and stop
        R, t = look_at(s * end * target[2], target)
        view, _ = render(points, colors, K, R, t, image.shape[:2])
        if s > 0.02:
            draw_camera(view, K, R, t, K, image.shape[:2], 0.25 * target[2], color=(220, 30, 30))
        video.append(view)
    return video[:1] * hold + video + video[-1:] * hold + video[::-1]

lift = lift_video('quad', np.array([0.35, -0.8, -0.6]))
fig, axs = plt.subplots(1, 2, figsize=(18, 6.5))
for ax, frame, title in zip(axs, [lift[0], lift[len(lift) // 2]], ['the photo', 'the same points, seen from above']):
    ax.imshow(frame); ax.set_title(title, fontsize=15); ax.axis('off')
savefig('3d_lift')
plt.show()
save_video(lift, '3d_lift', fps=15)

### Holes in novel views

Now we orbit the camera around the pots. Wherever the new camera looks behind something, no point lands: the holes (magenta) are the parts of the scene the photo never saw. Filling them in, plausibly and consistently, is what a generative model or world model has to do.

In [ ]:
def orbit(angle_degrees, pivot_depth, up=0.0):
    """camera rotated by `angle` around a vertical axis through the point at `pivot_depth` in front of the original camera"""
    a = np.radians(angle_degrees)
    R = np.array([[np.cos(a), 0, -np.sin(a)], [0, 1, 0], [np.sin(a), 0, np.cos(a)]])
    pivot = np.array([0, 0, pivot_depth])
    return R, -R @ pivot + pivot + np.array([0, up, 0])

def holes_view(name, angle, up=0.0):
    points, colors, K = clouds[name]
    R, t = orbit(angle, np.median(points[:, 2]), up)
    view, hit = render(points, colors, K, R, t, images[name].shape[:2], background=(255, 0, 255))
    return view, 100 * (1 - hit.mean())

views = [('original view', 0), ('orbit 10° left', -10), ('orbit 10° right', 10), ('orbit 20° right', 20)]
fig, axs = plt.subplots(2, 2, figsize=(16, 11))
for ax, (title, angle) in zip(axs.ravel(), views):
    view, holes = holes_view('pots', angle)
    ax.imshow(view)
    ax.set_title(f'{title}: {holes:.0f}% of the pixels are holes', fontsize=15)
    ax.axis('off')
plt.tight_layout()
savefig('3d_holes')
plt.show()

video = []
for angle in 20 * np.sin(np.linspace(0, 2 * np.pi, 90, endpoint=False)):
    view, holes = holes_view('pots', angle)
    cv2.putText(view, f'{angle:+.0f} deg, holes: {holes:.0f}%', (16, 40), cv2.FONT_HERSHEY_SIMPLEX, 1.1, (0, 0, 0), 5, cv2.LINE_AA)
    cv2.putText(view, f'{angle:+.0f} deg, holes: {holes:.0f}%', (16, 40), cv2.FONT_HERSHEY_SIMPLEX, 1.1, (255, 255, 255), 2, cv2.LINE_AA)
    video.append(view)
save_video(video, '3d_holes', fps=15)

## A toy world model

The world: a 64 x 64 room seen from above. We control the red agent: the action $a_t \in [-1, 1]^2$ moves it by up to 3 pixels in x and y (walls stop it). A blue ball we do *not* control comes down the middle, and at the junction it turns left or right **at random**, then starts again at the top. The coin flip is there on purpose: it is the smallest example of a future that cannot be predicted from the past (a pedestrian who may or may not cross).

The *state* of this world is the agent's and the ball's positions and the ball's coin flip; the *observation* is the 64 x 64 image. A world model predicts the next observation from the past observations and the action, $\hat o_{t+1} = f(o_{t-1}, o_t, a_t)$. Two frames are enough here: they show where things are and how fast they move.

In [ ]:
torch.backends.cudnn.benchmark = True
H = W = 64
AGENT_R, BALL_R = 5.0, 4.0
LO, HI = 9.5, 54.5             # the agent's centre stays inside the walls
SPEED = 3.0                    # pixels per step at |a| = 1
DOWN, SIDE = 4.0, 6.0          # the ball's speed down the middle and to the side
JUNCTION = 7                   # steps down the middle; then 4 steps to the left or right, then it starts again
CYCLE = JUNCTION + 4

yy, xx = torch.meshgrid(torch.arange(H, device=device) + 0.5, torch.arange(W, device=device) + 0.5, indexing='ij')
WALLS = ((xx < 4) | (xx > W - 4) | (yy < 4) | (yy > H - 4)).float()
COLOURS = {name: torch.tensor(c, device=device)[:, None, None] for name, c in
           [('floor', [0.93, 0.92, 0.87]), ('wall', [0.35, 0.35, 0.38]), ('agent', [0.85, 0.15, 0.15]), ('ball', [0.15, 0.35, 0.85])]}

def disc(cx, cy, r):
    """anti-aliased discs, one per batch element: (B,) centres -> (B, 1, H, W) masks"""
    d = torch.sqrt((xx - cx[:, None, None]) ** 2 + (yy - cy[:, None, None]) ** 2)
    return (r + 0.5 - d).clamp(0, 1)[:, None]

class Room:
    """B rooms simulated in parallel on the GPU"""
    def __init__(self, B, gen, phase=None):
        self.gen = gen
        self.agent = LO + (HI - LO) * torch.rand(B, 2, device=device, generator=gen)
        self.phase = torch.randint(0, CYCLE, (B,), device=device, generator=gen) if phase is None else torch.full((B,), phase, device=device)
        self.side = self.coin(B)

    def coin(self, B):
        return torch.randint(0, 2, (B,), device=device, generator=self.gen).float() * 2 - 1

    def ball(self):
        down = self.phase < JUNCTION
        j = (self.phase - JUNCTION + 1).clamp(min=0).float()
        x = torch.where(down, torch.full_like(j, 32.0), 32.0 + self.side * SIDE * j)
        y = torch.where(down, 10.0 + DOWN * self.phase.float(), torch.full_like(j, 10.0 + DOWN * (JUNCTION - 1)))
        return x, y

    def render(self):
        B = len(self.agent)
        img = COLOURS['floor'] * (1 - WALLS) + COLOURS['wall'] * WALLS
        img = img.expand(B, 3, H, W)
        m = disc(self.agent[:, 0], self.agent[:, 1], AGENT_R)
        img = img * (1 - m) + COLOURS['agent'] * m
        m = disc(*self.ball(), BALL_R)
        return img * (1 - m) + COLOURS['ball'] * m

    def step(self, a):
        self.agent = (self.agent + SPEED * a.clamp(-1, 1)).clamp(LO, HI)
        self.phase = (self.phase + 1) % CYCLE
        turn = self.phase == JUNCTION                  # the ball reaches the junction: left or right at random
        self.side = torch.where(turn, self.coin(len(turn)), self.side)
        return self.render()

def random_actions(B, T, gen):
    """smoothly varying random actions, like someone exploring the room"""
    a, out = torch.zeros(B, 2, device=device), []
    for _ in range(T):
        a = (0.7 * a + 0.6 * torch.randn(B, 2, device=device, generator=gen)).clamp(-1, 1)
        out.append(a)
    return torch.stack(out, 1)

def simulate(B, T, gen, phase=None, actions=None):
    """frames (B, T+1, 3, H, W) and actions (B, T, 2); actions[:, k] takes frames[:, k] to frames[:, k+1]"""
    room = Room(B, gen, phase)
    actions = random_actions(B, T, gen) if actions is None else actions
    frames = [room.render()] + [room.step(actions[:, k]) for k in range(T)]
    return torch.stack(frames, 1), actions

def show(frame):
    return frame.clamp(0, 1).permute(1, 2, 0).cpu().numpy()

JUNCTION_Y = 10.0 + DOWN * (JUNCTION - 1)
def ball_track(ax):
    """the ball's possible paths, drawn faintly for us (the model never sees them)"""
    ax.plot([32, 32], [10, JUNCTION_Y], color='0.6', linestyle=':', linewidth=2)
    ax.plot([32 - 3 * SIDE, 32 + 3 * SIDE], [JUNCTION_Y, JUNCTION_Y], color='0.6', linestyle=':', linewidth=2)

# four episodes with random actions; the arrow is the action
gen = torch.Generator(device=device).manual_seed(0)
T = 44
frames, actions = simulate(4, T, gen)
fig, axs = plt.subplots(2, 2, figsize=(8, 8.4), dpi=80)
video = []
for k in range(T):
    for b, ax in enumerate(axs.ravel()):
        ax.clear()
        ax.imshow(show(frames[b, k]), interpolation='nearest', extent=(0, W, H, 0))
        ball_track(ax)
        ax.axis('off')
    for b, ax in enumerate(axs.ravel()):
        # agent position from the frame (red pixels), the action that comes next as an arrow
        red = (frames[b, k, 0] - 0.5 * (frames[b, k, 1] + frames[b, k, 2])).clamp(min=0)
        cx, cy = (red * xx).sum() / red.sum(), (red * yy).sum() / red.sum()
        dx, dy = 3 * SPEED * actions[b, k].cpu().numpy()
        ax.annotate('', (cx.item() + dx, cy.item() + dy), (cx.item(), cy.item()),
                    arrowprops=dict(arrowstyle='-|>', color='k', lw=2.5, mutation_scale=18))
    fig.suptitle(f'four episodes, t = {k}', fontsize=18)
    video.append(grab(fig))
plt.close(fig)
save_video(video, 'wm_toy', fps=4, width=500)

### Two futures

The same past and the same action (here: standing still) can lead to two different futures: the ball turns left or right. Below we run the simulator twice from exactly the same state and force the coin flip.

In [ ]:
gen = torch.Generator(device=device).manual_seed(3)
room = Room(2, gen, phase=JUNCTION - 3)
room.agent[1] = room.agent[0]
still = torch.zeros(2, 2, device=device)
seq = [room.render()]
for k in range(5):
    seq.append(room.step(still))
    if room.phase[0] == JUNCTION:
        room.side = torch.tensor([-1.0, 1.0], device=device)    # future A turns left, future B right
        seq[-1] = room.render()
panels = [('observed $o_{t-1}$', seq[1][0]), ('observed $o_t$', seq[2][0]),
          ('future A: $o_{t+3}$', seq[5][0]), ('future B: $o_{t+3}$', seq[5][1])]
fig, axs = plt.subplots(2, 2, figsize=(9, 9.6))
for ax, (title, img) in zip(axs.ravel(), panels):
    ax.imshow(show(img), interpolation='nearest')
    ax.axvline(31.5, color='k', linestyle=':', linewidth=1.5)
    ax.set_title(title, fontsize=18)
    ax.axis('off')
plt.tight_layout(w_pad=2, h_pad=4)
savefig('wm_futures')
plt.show()

### The model

A small U-Net (Lecture 11) takes the two frames and predicts the change to the last one; the action enters every block as a learned per-channel bias (as the time step does in a diffusion model, Lecture 16). We train it one step ahead with the squared error, on random episodes generated on the fly (about a minute on a T4).

In [ ]:
class Block(nn.Module):
    def __init__(self, c_in, c_out, c_cond):
        super().__init__()
        self.conv1, self.conv2 = nn.Conv2d(c_in, c_out, 3, padding=1), nn.Conv2d(c_out, c_out, 3, padding=1)
        self.norm1, self.norm2 = nn.GroupNorm(8, c_out), nn.GroupNorm(8, c_out)
        self.cond = nn.Linear(c_cond, c_out)

    def forward(self, x, c):
        h = F.silu(self.norm1(self.conv1(x))) + self.cond(c)[:, :, None, None]
        return F.silu(self.norm2(self.conv2(h)))

class UNet(nn.Module):
    def __init__(self, c_in, c_cond, c_out=3, ch=(32, 64, 128), c_emb=128):
        super().__init__()
        self.emb = nn.Sequential(nn.Linear(c_cond, c_emb), nn.SiLU(), nn.Linear(c_emb, c_emb))
        self.down = nn.ModuleList([Block(a, b, c_emb) for a, b in zip((c_in,) + ch[:-1], ch)])
        self.mid = Block(ch[-1], ch[-1], c_emb)
        self.up = nn.ModuleList([Block(2 * b, a, c_emb) for a, b in zip((ch[0],) + ch[:-1], ch)][::-1])
        self.out = nn.Conv2d(ch[0], c_out, 1)

    def forward(self, x, cond):
        c, skips = self.emb(cond), []
        for block in self.down:
            x = block(x, c)
            skips.append(x)
            x = F.avg_pool2d(x, 2)
        x = self.mid(x, c)
        for block in self.up:
            x = block(torch.cat([F.interpolate(x, scale_factor=2), skips.pop()], 1), c)
        return self.out(x)

class Predictor(nn.Module):
    """deterministic world model: o_{t+1} = o_t + UNet(o_{t-1}, o_t; a_t)"""
    def __init__(self):
        super().__init__()
        self.net = UNet(6, 2)

    def forward(self, prev, cur, a):
        return cur + self.net(torch.cat([prev, cur], 1), a)

def train_predictor(unroll=1, steps=2500, B=64, seed=0, init=None, lr=2e-3):
    """unroll = 1: one step ahead on real frames. unroll > 1: the model is fed its own predictions for `unroll` steps
    and the loss is averaged over all of them (backpropagated through the whole rollout). init: start from a copy of
    another model's weights (fine-tuning)"""
    torch.manual_seed(seed)
    gen = torch.Generator(device=device).manual_seed(seed)
    model = Predictor().to(device)
    if init is not None:
        model.load_state_dict(init.state_dict())
    opt = torch.optim.AdamW(model.parameters(), lr=lr)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, total_steps=steps)
    for _ in tqdm(range(steps)):
        frames, actions = simulate(B, unroll + 1, gen)
        prev, cur, loss = frames[:, 0], frames[:, 1], 0
        for k in range(unroll):
            pred = model(prev, cur, actions[:, k + 1])
            loss = loss + F.mse_loss(pred, frames[:, k + 2]) / unroll
            prev, cur = cur, pred
        opt.zero_grad()
        loss.backward()
        opt.step()
        sched.step()
    return model.eval()

one_step = train_predictor(unroll=1)

## The blurred future

Just before the junction, the ball's next position is a coin flip. The squared error is minimised by the **mean** of the possible futures, so the regression model draws two half-transparent ghost balls, a future that never happens (the same effect as the brownish mean colour in Lecture 17 and the blurry few-step samples in Lecture 16).

A generative model instead learns the *distribution* of next frames. We train a conditional flow-matching model (Lecture 16) for the change to the last frame: the same U-Net, with the noisy sample $x_\tau$ as an extra input and the flow time $\tau$ as an extra condition. The network predicts the end point $\hat x_1$ of the straight path, and the flow direction follows as $(\hat x_1 - x_\tau) / (1 - \tau)$; this keeps the large, empty background exactly as it is. Each sample picks one branch, sharply.

In [ ]:
class FlowNext(nn.Module):
    """conditional flow matching for the change to the next frame, x1 = o_{t+1} - o_t, given (o_{t-1}, o_t, a_t);
    the network predicts x1 (the end of the straight path) from the noisy x at time tau"""
    def __init__(self):
        super().__init__()
        self.net = UNet(9, 2 + 9)

    def forward(self, x, tau, prev, cur, a):
        k = torch.arange(1, 5, device=device) * np.pi
        t_emb = torch.cat([tau[:, None], torch.sin(tau[:, None] * k), torch.cos(tau[:, None] * k)], 1)
        return self.net(torch.cat([x, prev, cur], 1), torch.cat([a, t_emb], 1))

    @torch.no_grad()
    def sample(self, prev, cur, a, steps=30, gen=None):
        x = torch.randn(cur.shape, device=device, generator=gen)
        for i in range(steps):                                     # Euler's method from tau = 0 (noise) to 1
            tau = torch.full((len(x),), i / steps, device=device)
            v = (self.forward(x, tau, prev, cur, a) - x) / (1 - tau[:, None, None, None])
            x = x + v / steps
        return cur + x

def train_flow(steps=6000, B=64, seed=0):
    torch.manual_seed(seed)
    gen = torch.Generator(device=device).manual_seed(seed + 1)
    model = FlowNext().to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=2e-3)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=2e-3, total_steps=steps)
    for _ in tqdm(range(steps)):
        # half of each batch starts just before the junction, so the ambiguous case is well represented
        frames, actions = simulate(B, 2, gen)
        f2, a2 = simulate(B // 2, 2, gen, phase=JUNCTION - 2)
        frames, actions = torch.cat([frames[B // 2:], f2]), torch.cat([actions[B // 2:], a2])
        x1 = frames[:, 2] - frames[:, 1]
        x0 = torch.randn(x1.shape, device=device, generator=gen)
        tau = torch.rand(len(x1), device=device, generator=gen)
        x = (1 - tau[:, None, None, None]) * x0 + tau[:, None, None, None] * x1
        loss = F.mse_loss(model(x, tau, frames[:, 0], frames[:, 1], actions[:, 1]), x1)
        opt.zero_grad()
        loss.backward()
        opt.step()
        sched.step()
    return model.eval()

flow = train_flow()

# one situation just before the junction, standing still (a = 0): regression vs. samples
gen = torch.Generator(device=device).manual_seed(3)
frames, actions = simulate(1, 2, gen, phase=JUNCTION - 2, actions=torch.zeros(1, 2, 2, device=device))
prev, cur, a = frames[:, 0], frames[:, 1], actions[:, 1]
with torch.no_grad():
    mean = one_step(prev, cur, a)
gen = torch.Generator(device=device).manual_seed(0)
samples = flow.sample(prev.expand(16, -1, -1, -1), cur.expand(16, -1, -1, -1), a.expand(16, -1), gen=gen)
ball_x = [(s[2, 30:40] - s[0, 30:40]).clamp(min=0).sum(0).argmax().item() for s in samples]   # where is the blue ball?
left = [i for i, x in enumerate(ball_x) if x < 32]
right = [i for i, x in enumerate(ball_x) if x >= 32]
print(f'{len(left)} of {len(samples)} samples turn left, {len(right)} turn right')
pick = (left[:1] + right[:1] + left[1:2] + right[1:2] + left + right)[:3]

panels = [('observed $o_{t-1}$', prev[0]), ('observed $o_t$', cur[0]), ('regression (L2): the mean', mean[0])] + \
         [(f'flow matching: sample {k + 1}', samples[i]) for k, i in enumerate(pick)]
fig, axs = plt.subplots(2, 3, figsize=(13, 9.6))
for ax, (title, img) in zip(axs.ravel(), panels):
    ax.imshow(show(img), interpolation='nearest')
    ax.axvline(31.5, color='k', linestyle=':', linewidth=1.5)     # the junction: left or right of it?
    ax.set_title(title, fontsize=17)
    ax.axis('off')
plt.tight_layout(w_pad=2, h_pad=4)
savefig('wm_blur')
plt.show()

## Rolling out: errors compound

To look further ahead, the model is fed its own predictions: $\hat o_{t+2} = f(o_t, \hat o_{t+1}, a_{t+1})$, and so on. The one-step model was only ever trained on real frames; its own predictions are slightly wrong in ways it has never seen, and the errors grow with every step. Fine-tuning the model on its own rollouts (here 6 steps, backpropagating through all of them) teaches it to recover from its mistakes.

Both models are deterministic, so after the ball's coin flip neither can be right about the ball: part of the error is the blurred future again.

In [ ]:
multi_step = train_predictor(unroll=6, steps=800, init=one_step, lr=5e-4, seed=1)   # fine-tune the one-step model

@torch.no_grad()
def rollout(model, frames, actions):
    """feed the model its own predictions, starting from the first two real frames"""
    prev, cur, out = frames[:, 0], frames[:, 1], [frames[:, 1]]
    for k in range(1, actions.shape[1]):
        prev, cur = cur, model(prev, cur, actions[:, k])
        out.append(cur)
    return torch.stack(out, 1)

T = 31
gen = torch.Generator(device=device).manual_seed(1)
frames, actions = simulate(256, T, gen)
preds = {'trained one step ahead': rollout(one_step, frames, actions),
         'fine-tuned on 6-step rollouts': rollout(multi_step, frames, actions)}
errors = {name: ((p - frames[:, 1:]) ** 2).mean((0, 2, 3, 4)).cpu().numpy() for name, p in preds.items()}

steps_shown = [1, 5, 10, 20, 30]
i = 0                                                          # one of the 256 rollouts
fig = plt.figure(figsize=(24, 8.5))
gs = fig.add_gridspec(3, len(steps_shown) + 2, width_ratios=[1] * len(steps_shown) + [0.55, 3.2], wspace=0.08, hspace=0.12)
rows = [('ground truth', frames[i, 1:])] + [(name, p[i]) for name, p in preds.items()]
for r, (name, seq) in enumerate(rows):
    for c, k in enumerate(steps_shown):
        ax = fig.add_subplot(gs[r, c])
        ax.imshow(show(seq[k]), interpolation='nearest')
        ax.set_xticks([]); ax.set_yticks([])
        if r == 0:
            ax.set_title(f'{k} step{"s" if k > 1 else ""} ahead', fontsize=15)
        if c == 0:
            ax.set_ylabel(name.replace(' on ', '\non ').replace(' one', '\none'), fontsize=13)
ax = fig.add_subplot(gs[:, -1])
for name, e in errors.items():
    ax.plot(np.arange(T), e, linewidth=3, label=name)
ax.set_xlabel('steps ahead', fontsize=15)
ax.set_ylabel('mean squared error', fontsize=15)
ax.tick_params(labelsize=13)
ax.legend(fontsize=14)
ax.grid(alpha=0.3)
ax.set_title('error of 256 rollouts', fontsize=15)
savefig('wm_rollout')
plt.show()

# the error curves alone (for the slide), and the rollout as a video
fig, ax = plt.subplots(figsize=(7, 6))
for name, e in errors.items():
    ax.plot(np.arange(T), e, linewidth=3, label=name)
ax.set_xlabel('steps ahead', fontsize=16)
ax.set_ylabel('mean squared error', fontsize=16)
ax.tick_params(labelsize=13)
ax.legend(fontsize=14)
ax.grid(alpha=0.3)
ax.set_title('error, averaged over 256 rollouts', fontsize=16)
savefig('wm_rollout_error')
plt.close(fig)

fig, axs = plt.subplots(1, 3, figsize=(13.5, 5.2), dpi=80)
video = []
for k in range(T):
    for ax, (name, seq) in zip(axs, [('real room', frames[i, 1:])] + [(name, p[i]) for name, p in preds.items()]):
        ax.clear()
        ax.imshow(show(seq[k]), interpolation='nearest')
        ax.set_title(name, fontsize=17)
        ax.axis('off')
    fig.suptitle(f'{k} step{"s" if k != 1 else ""} ahead (same actions in all three)', fontsize=19)
    video.append(grab(fig))
plt.close(fig)
save_video(video, 'wm_rollout_video', fps=3)

## Planning with a world model

The model lets us try actions *in imagination* before acting. Given a goal image, model-predictive control with the cross-entropy method (CEM):

1. sample 128 sequences of 8 actions from a Gaussian,
2. roll each one out with the world model and score the last predicted frame against the goal image,
3. refit the Gaussian to the 16 best sequences and repeat (4 times),
4. execute only the first action in the real room, observe, and plan again.

The score compares *features* of the predicted frame and the goal image, not raw pixels: the ball is somewhere else in the goal image, and that difference would drown the signal. Our feature is hand-made (how red each pixel is), blurred at two scales so that "close to the goal" counts even when the discs do not overlap. The planner never sees coordinates, only images. This is how V-JEPA 2 plans robot actions, with features learned from video instead of our redness map.

In the video, the dotted line is the plan the model *imagines* at each step (where the red disc is in its predicted frames); the solid line is the path actually taken.

In [ ]:
def blur(x, sigma):
    """separable Gaussian blur of a one-channel image (zero outside the image)"""
    r = int(3 * sigma)
    k = torch.exp(-torch.arange(-r, r + 1, device=device).float() ** 2 / (2 * sigma ** 2))
    k = (k / k.sum()).view(1, 1, 1, -1)
    x = F.conv2d(F.pad(x, (r, r, 0, 0)), k)
    return F.conv2d(F.pad(x, (0, 0, r, r)), k.transpose(2, 3))

def features(x):
    """a hand-made feature map: how red each pixel is (only the agent is red)"""
    return (x[:, :1] - 0.5 * (x[:, 1:2] + x[:, 2:3])).clamp(min=0)

def cost(frames, goal):
    """distance between the feature maps of the predicted frames and the goal image, blurred at two scales"""
    d = features(frames) - features(goal)
    return sum((blur(d, s) ** 2).mean((1, 2, 3)) for s in (4, 12))

@torch.no_grad()
def cem(model, prev, cur, goal, gen, horizon=8, samples=128, elites=16, iterations=4):
    """returns the planned action sequence (horizon, 2)"""
    mu, std = torch.zeros(horizon, 2, device=device), torch.full((horizon, 2), 0.8, device=device)
    for _ in range(iterations):
        a = (mu + std * torch.randn(samples, horizon, 2, device=device, generator=gen)).clamp(-1, 1)
        p, c = prev.expand(samples, -1, -1, -1), cur.expand(samples, -1, -1, -1)
        for k in range(horizon):
            p, c = c, model(p, c, a[:, k])
        best = a[cost(c, goal).argsort()[:elites]]
        mu, std = best.mean(0), best.std(0) + 0.05
    return mu

@torch.no_grad()
def imagine(model, prev, cur, plan):
    """where the red disc is in the model's predicted frames for a plan (its centre of 'redness')"""
    out = []
    for a in plan:
        prev, cur = cur, model(prev, cur, a[None])
        f = features(cur)[0, 0]
        out.append(((f * xx).sum() / f.sum()).item())
        out.append(((f * yy).sum() / f.sum()).item())
    return np.array(out).reshape(-1, 2)

def goal_image(goal_xy):
    room = Room(1, torch.Generator(device=device).manual_seed(0), phase=0)
    room.agent = torch.tensor([goal_xy], device=device, dtype=torch.float)
    return room.render()

tasks = [((12, 52), (50, 14)), ((50, 50), (14, 20)), ((30, 12), (34, 52))]
runs = []
for start, goal_xy in tasks:
    gen = torch.Generator(device=device).manual_seed(0)
    room = Room(1, gen, phase=0)
    room.agent = torch.tensor([start], device=device, dtype=torch.float)
    goal = goal_image(goal_xy)
    prev = cur = room.render()
    run = dict(start=start, goal=goal_xy, path=[room.agent[0].tolist()], frames=[cur[0]], plans=[])
    for t in range(50):
        plan = cem(multi_step, prev, cur, goal, gen)
        run['plans'].append(imagine(multi_step, prev, cur, plan))
        prev, cur = cur, room.step(plan[:1])
        run['path'].append(room.agent[0].tolist())
        run['frames'].append(cur[0])
        if np.hypot(*(np.array(run['path'][-1]) - goal_xy)) < 3:
            break
    runs.append(run)

def draw_planning(ax, run, k, show_plan):
    ax.imshow(show(run['frames'][k]), interpolation='nearest', extent=(0, W, H, 0))
    p = np.array(run['path'][:k + 1])
    ax.plot(p[:, 0], p[:, 1], 'k.-', linewidth=2, markersize=6)
    if show_plan and k < len(run['plans']):
        q = np.vstack([p[-1:], run['plans'][k]])
        ax.plot(q[:, 0], q[:, 1], ':', color='0.25', linewidth=2.5)
        ax.plot(q[1:, 0], q[1:, 1], 'o', color='0.25', markersize=4, fillstyle='none')
    ax.add_patch(plt.Circle(run['goal'], AGENT_R + 1, fill=False, color='k', linestyle='--', linewidth=2.5))
    ax.plot(*run['start'], 'ks', markersize=10)
    ax.axis('off')

fig, axs = plt.subplots(1, len(runs), figsize=(24, 8.6))
for ax, run in zip(axs, runs):
    n = len(run['path']) - 1
    draw_planning(ax, run, n, False)
    miss = np.hypot(*(np.array(run['path'][-1]) - run['goal']))
    ax.set_title(f'goal (dashed) reached in {n} steps' if miss < 3 else f'after {n} steps: {miss:.0f} px from the goal', fontsize=17)
plt.tight_layout(w_pad=4)
savefig('wm_planning')
plt.show()

fig, axs = plt.subplots(1, len(runs), figsize=(15, 5.6), dpi=80)
video = []
for k in range(max(len(r['path']) for r in runs) + 4):
    for ax, run in zip(axs, runs):
        ax.clear()
        n = len(run['path']) - 1
        draw_planning(ax, run, min(k, n), k < n)
        reached = np.hypot(*(np.array(run['path'][-1]) - run['goal'])) < 3
        ax.set_title(f'step {min(k, n)}' + (': goal reached' if k >= n and reached else ''), fontsize=17)
    fig.suptitle('dotted: the plan the model imagines; solid: the path taken; dashed: the goal', fontsize=17)
    video.append(grab(fig))
plt.close(fig)
save_video(video, 'wm_planning_video', fps=3)